# 01 - Data overview

**Purpose:** Summarise the real datasets read-only and chart the banking PQC observations by scan date (Track C, descriptive only - no model).

**Inputs:** CESNET-TLS22 S (HDF5, via env vars / default pack path); banking_pqc_observations.csv

**Outputs:** inline tables and chart; scratch/derived/member3/figs/pqc_status_by_date.png

**Status labels:** VERIFIED (real public data, read-only). Not bank data; ERROR stays unclassified; no training on status labels.

TEST/demo context: the demo bank in `demo_bank_app/` is SYNTHETIC.

In [ ]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "cryptoshift").is_dir())
for p in (str(ROOT / "src"), str(ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)
DERIVED = ROOT / "scratch" / "derived" / "member3"
DERIVED.mkdir(parents=True, exist_ok=True)
# This notebook reads no credentials. If a future cell needs one, read it from an environment variable at runtime
# and never display it.

In [ ]:
from cryptoshift.data_summary import build_data_summary
s = build_data_summary(chunk_size=20_000, max_stream_rows=40_000)
pqc, ces = s['banking_pqc_observations'], s['cesnet_tls22_s']
print(pqc['row_count'], 'rows |', pqc['scan_dates'], '| duplicates', pqc['duplicate_observation_keys'], '| status', pqc['status_counts'])
print(ces.get('total_rows'), 'CESNET rows |', ces.get('table_count'), 'tables |', ces.get('date_range'))

In [ ]:
import pandas as pd
share = pd.DataFrame(pqc['status_counts_by_scan_date']).T.fillna(0)
share = share.div(share.sum(axis=1), axis=0).round(3)
share  # ERROR kept as its own (unclassified) category; repeated hosts across dates are kept

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
ax = share[['PQ-SAFE','CLASSICAL','ERROR']].plot(kind='bar', stacked=True, figsize=(7,3.5), title='Banking public-endpoint observations by scan date (share)')
ax.set_ylabel('share of 64 hosts'); plt.tight_layout()
(DERIVED / 'figs').mkdir(exist_ok=True); plt.savefig(DERIVED / 'figs' / 'pqc_status_by_date.png', dpi=120)

Public-endpoint observations on the given dates only. They say nothing about a bank's internal systems, signatures or migration status.